# Etap 1 — Wydobycie, oczyszczenie i podział regulaminu na fragmenty

Projekt **RAG: „Czatbot odpowiadający na pytania o regulamin studiów”**.

Realizowana rola: *„Wydobycie i oczyszczenie tekstu regulaminu, podział na fragmenty z numerami paragrafów i ustępów”*.

Ten notebook wykonuje kolejno:

1. **Wydobycie i oczyszczenie** tekstu z pliku `regulamin.pdf`,
2. **Podział semantyczny** na fragmenty (1 fragment = 1 ustęp wraz z podpunktami `a.`, `b.`, `c.`) z metadanymi `paragraf` / `ustęp`,
3. **Analizę tokenów** fragmentów przy użyciu biblioteki `tiktoken`,
4. **Zapis** wyników do pliku `fragmenty_regulaminu.json`.


In [ ]:
# Zależność: pip install pymupdf  (biblioteka PyMuPDF, importowana jako fitz/pymupdf)
try:
    import pymupdf as fitz   # nowy sposób importu (PyMuPDF >= 1.24)
except ImportError:
    import fitz              # starszy sposób importu
import re


def extract_and_clean_pdf(pdf_path, output_path):
    """
    Wyciąga tekst z PDF (od strony 3) i oczyszcza go:
    - pomija pierwsze 2 strony uchwały,
    - usuwa numery stron,
    - skleja wyrazy przeniesione myślnikiem,
    - naprawia przełamane wiersze w środku zdania,
    - zamienia uszkodzone tabele (§ 18 ust. 10 i § 32 ust. 6) na Markdown.
    """
    doc = fitz.open(pdf_path)
    raw_text = ""
    for page_num in range(2, len(doc)):          # indeks 2 = strona 3
        raw_text += doc[page_num].get_text("text")

    cleaned_text = raw_text

    # 1) Numery stron: wzorce typu "- 7 -"
    cleaned_text = re.sub(r'-\s*\d+\s*-', '', cleaned_text)

    # 2) Wyrazy przeniesione: "infor-\nmacja" -> "informacja"
    cleaned_text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', cleaned_text)

    # 3) Przełamane wiersze w środku zdania (mała litera przed i po nowej linii)
    cleaned_text = re.sub(r'(?<=[a-ząćęłńóśźż,])\n(?=[a-ząćęłńóśźż])', ' ', cleaned_text)

    # 4) Wielokrotne puste linie -> najwyżej jedna
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)

    # 5) Naprawa tabel (zastąpienie "papki" z PDF czystym Markdownem)
    tabela_18_10 = """
| Efekt uczenia się | Na ocenę 2 | Na ocenę 3 | Na ocenę 3.5 | Na ocenę 4 | Na ocenę 4.5 | Na ocenę 5 |
|---|---|---|---|---|---|---|
| Wiedza | 0-59,99% | 60-74,99% | 75-79,99% | 80-88,99% | 89-94,99% | 95-100% |
| Umiejętności | Nie posiada wymienionych umiejętności | W dostateczny sposób opanował wymienione umiejętności | Dość dobrze opanował wymienione umiejętności | Dobrze opanował wymienione umiejętności | W znacznym stopniu opanował wymienione umiejętności | Bardzo dobrze opanował wymienione umiejętności |
| Kompetencje społeczne | Nie posiada wymienionych kompetencji | W dostateczny sposób opanował wymienione kompetencje | Dość dobrze opanował wymienione kompetencje | Dobrze opanował wymienione kompetencje | W znacznym stopniu opanował wymienione kompetencje | Bardzo dobrze opanował wymienione kompetencje |
""".strip()

    tabela_32_6 = """
| Obliczona wartość | Oceny po zaokrągleniu |
|---|---|
| 2,00-2,99 | 2,0 |
| 3,00-3,25 | 3,0 |
| 3,26-3,75 | 3,5 |
| 3,76-4,25 | 4,0 |
| 4,26-4,75 | 4,5 |
| 4,76-5,00 | 5,0 |
""".strip()

    # Uwaga: dokument używa półpauzy "–" (U+2013); wzorzec [–-] dopuszcza
    # zarówno półpauzę, jak i zwykły myślnik.
    cleaned_text = re.sub(
        r'Efekt\s+uczenia\s+się.*?Bardzo\s+dobrze\s+opanował\s+wymienione\s+kompetencje',
        tabela_18_10, cleaned_text, flags=re.DOTALL)
    cleaned_text = re.sub(
        r'Obliczona\s+wartość.*?4,76[–-]5,00\s+5,0',
        tabela_32_6, cleaned_text, flags=re.DOTALL)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(cleaned_text)

    return raw_text, cleaned_text


# Wykonanie czyszczenia
pdf_file = "regulamin.pdf"
txt_file = "regulamin_oczyszczony.txt"
raw_text, clean_text = extract_and_clean_pdf(pdf_file, txt_file)


# Podgląd "przed / po" na przykładzie tabeli § 32 ust. 6
def pokaz_okolicy(text, znacznik, przed=20, po=260):
    idx = text.find(znacznik)
    if idx == -1:
        return f"(nie znaleziono: {znacznik})"
    return text[max(0, idx - przed): idx + po]


print("=== PRZED (surowa tabela § 32 ust. 6 z PDF) ===")
print(pokaz_okolicy(raw_text, "Obliczona wartość"))
print("\n=== PO (tabela w formacie Markdown) ===")
print(pokaz_okolicy(clean_text, "| Obliczona wartość"))


### Co robi powyższa komórka i dlaczego?

Czysty, ustrukturyzowany tekst to podstawa dalszych kroków (podziału na fragmenty i osadzania). Krok po kroku:

- **Pomijamy 2 pierwsze strony** — to uchwała Senatu, a nie treść regulaminu.
- **Usuwamy numery stron** (wzorce `- N -`) — to szum, który zakłóciłby podział.
- **Sklejamy wyrazy przeniesione** (`infor-\nmacja`) — bez tego tokenizer policzyłby dwa „urwane” wyrazy.
- **Naprawiamy przełamane wiersze** w środku zdania — przywracamy ciągłość zdań.
- **Zamieniamy uszkodzone tabele na Markdown** — w PDF-ie tabele (§ 18 ust. 10 i § 32 ust. 6) są „rozsypane” na pojedyncze wiersze; zapis Markdown (`| ... |`) zachowuje ich strukturę i jest czytelny dla modelu.

Dzięki temu dalszy parser widzi spójny tekst, w którym nagłówki `§ N.` i numeracja ustępów `N.` są jednoznaczne.


In [ ]:
import re


# Tytuł rozdziału: "N. Tytuł" — po numerze wyłącznie małe litery (brak wielkich
# liter w środku i interpunkcji), np. "2. Organizacja studiów". Pozwala to odróżnić
# tytuł rozdziału od ustępu typu "5. Przyjęcie na studia ... Senat Uczelni.".
WZORZEC_ROZDZIALU = re.compile(
    r'^\d+\.\s+[A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż]*(?:\s+[a-ząćęłńóśźż]+)*\s*$'
)


def podziel_na_fragmenty(text):
    """
    Dzieli oczyszczony tekst semantycznie:
    1 fragment = 1 ustęp (wraz z podpunktami a., b., c., ...).

    Zwraca (fragmenty, anomalie), gdzie każdy fragment to słownik
    z metadanymi: paragraf, ustęp, zrodlo, tekst.
    """
    linie = text.split('\n')
    n = len(linie)

    fragmenty = []
    biezacy_paragraf = None
    biezacy_ustep = None
    bufor = []
    pierwszy_ustep_paragrafu = {}   # paragraf -> numer pierwszego ustępu
    paragraf_ma_ustep = set()

    def zamknij_ustep():
        nonlocal biezacy_ustep, bufor
        if biezacy_paragraf is None or biezacy_ustep is None:
            bufor = []
            return
        tekst = ' '.join(l.strip() for l in bufor if l.strip())
        tekst = re.sub(r'\s+', ' ', tekst).strip()
        if tekst:
            fragmenty.append({
                'paragraf': biezacy_paragraf,
                'ustep': biezacy_ustep,
                'zrodlo': f'§ {biezacy_paragraf} ust. {biezacy_ustep}',
                'tekst': tekst,
            })
        bufor = []

    def nastepna_niepusta(i):
        for j in range(i + 1, n):
            if linie[j].strip():
                return linie[j].strip()
        return None

    for i, surowa in enumerate(linie):
        linia = surowa.strip()
        if not linia:
            continue

        # 1) Nagłówek paragrafu: "§ N."
        m = re.match(r'^§\s*(\d+)\.?\s*$', linia)
        if m:
            zamknij_ustep()
            biezacy_paragraf = int(m.group(1))
            biezacy_ustep = None
            continue

        # 2) Tytuł rozdziału — ignorujemy
        nastepna = nastepna_niepusta(i)
        if WZORZEC_ROZDZIALU.match(linia) and nastepna and re.match(r'^§\s*\d+', nastepna):
            continue

        # 3) Nagłówek ustępu: "N. treść"
        m = re.match(r'^(\d{1,2})\.\s+(.*)$', linia)
        if m:
            zamknij_ustep()
            biezacy_ustep = int(m.group(1))
            if biezacy_paragraf not in paragraf_ma_ustep:
                pierwszy_ustep_paragrafu[biezacy_paragraf] = biezacy_ustep
                paragraf_ma_ustep.add(biezacy_paragraf)
            bufor.append(linia)
            continue

        # 4) Podpunkt: "a. treść" lub samo "a."
        if re.match(r'^[a-ząćęłńóśźż]\.\s*', linia) and biezacy_paragraf is not None:
            if biezacy_ustep is None:
                biezacy_ustep = 1
            bufor.append(linia)
            continue

        # 5) Kontynuacja tekstu bieżącego ustępu
        if biezacy_paragraf is not None:
            if biezacy_ustep is None:
                biezacy_ustep = 1  # paragraf bez numerowanych ustępów (np. § 1)
            bufor.append(linia)

    zamknij_ustep()

    # Wykrywanie anomalii numeracji: paragrafy, których ustępy nie zaczynają się od 1
    anomalie = {
        p: pierwszy_ustep_paragrafu[p]
        for p in sorted(pierwszy_ustep_paragrafu)
        if pierwszy_ustep_paragrafu[p] != 1
    }
    return fragmenty, anomalie


# Wykonanie podziału
with open('regulamin_oczyszczony.txt', encoding='utf-8') as f:
    tekst_regulaminu = f.read()

fragmenty, anomalie = podziel_na_fragmenty(tekst_regulaminu)

print('Liczba fragmentów (ustępów):', len(fragmenty))
if anomalie:
    print('Wykryte anomalie numeracji (paragraf -> pierwszy numer ustępu):')
    for p, pierwszy in anomalie.items():
        print(f'  § {p}: numeracja ustępów zaczyna się od {pierwszy} (a nie od 1)')
else:
    print('Brak anomalii numeracji.')


### Dlaczego dzielimy tekst semantycznie na ustępy?

W architekturze RAG fragment (chunk) to jednostka, którą system wyszukuje i podaje modelowi LLM jako kontekst. Wybór **podziału semantycznego na ustępy** ma kilka zalet:

1. **Precyzyjne wskazanie źródła.** Każdy fragment ma metadane `§ N` i `ust. M`, więc model może zacytować dokładnie: *„§ 30 ust. 6”*. Przy podziale po stałej liczbie znaków granice fragmentów przecinałyby przepisy w losowych miejscach i utrudniały cytowanie.

2. **Logiczne, samodzielne bloki.** Ustęp to naturalna, zamknięta jednostka regulacji (jedna norma). Dołączamy do niego podpunkty `a.`, `b.`, `c.`, bo bez nich norma byłaby niepełna — np. wyliczenie kar bez podpunktów traci sens.

3. **Spójność treści przy wyszukiwaniu.** Fragment obejmuje całą myśl, więc wektorowe podobieństwo zapytania do fragmentu jest bardziej wiarygodne — nie „urywamy” zdania w połowie.

4. **Odpowiednia wielkość.** Ustępy są z reguły krótkie (kilka–kilkadziesiąt tokenów), co mieści się w limitach kontekstu modeli osadzających i LLM.

**Uwaga techniczna:** numeracja ustępów nie jest idealna — w § 30 zaczyna się od 6 (błąd w dokumencie źródłowym). Kod to wykrywa i **zachowuje oryginalną numerację**, aby cytaty zgadzały się z dokumentem.


In [ ]:
import tiktoken

# Tokenizer zgodny z modelami OpenAI (np. text-embedding-3-*, GPT-4).
# Przy pierwszym uruchomieniu tiktoken może pobrać plik kodowania z internetu.
enc = tiktoken.get_encoding('cl100k_base')


def liczba_tokenow(tekst):
    return len(enc.encode(tekst))


# Obliczamy liczbę tokenów i dopisujemy ją do metadanych fragmentów
for frag in fragmenty:
    frag['tokeny'] = liczba_tokenow(frag['tekst'])

dlugosci = [f['tokeny'] for f in fragmenty]

print('=== Statystyki fragmentów (tokeny, cl100k_base) ===')
print('Łączna liczba fragmentów:', len(fragmenty))
print('Średnia długość fragmentu:', round(sum(dlugosci) / len(dlugosci), 1), 'tokenów')
print('Mediana długości:', int(sorted(dlugosci)[len(dlugosci) // 2]), 'tokenów')
print('Największa długość:', max(dlugosci), 'tokenów')
print('Najmniejsza długość:', min(dlugosci), 'tokenów')

# Trzy przykładowe fragmenty (różne przypadki)
print('\n=== Przykładowe fragmenty ===')

# a) paragraf bez numerowanych ustępów (§ 1) -> oznaczony jako ustęp 1
f1 = next(f for f in fragmenty if f['paragraf'] == 1)
print(f"\n[1] Paragraf bez numerowanych ustępów ({f1['zrodlo']}, {f1['tokeny']} tokenów):")
print(f"    {f1['tekst']}")

# b) fragment z podpunktami (§ 13 ust. 1)
f2 = next(f for f in fragmenty if f['paragraf'] == 13 and f['ustep'] == 1)
print(f"\n[2] Fragment z podpunktami ({f2['zrodlo']}, {f2['tokeny']} tokenów):")
print(f"    {f2['tekst']}")

# c) najdłuższy fragment
f3 = max(fragmenty, key=lambda f: f['tokeny'])
print(f"\n[3] Najdłuższy fragment ({f3['zrodlo']}, {f3['tokeny']} tokenów):")
print(f"    {f3['tekst']}")


### Co daje analiza tokenów?

- **Tokeny** to jednostki, na które tokenizer dzieli tekst (z grubsza: fragmenty wyrazów). Modele mają **limity tokenów** w oknie kontekstu, więc musimy wiedzieć, jak długie są nasze fragmenty.
- **Średnia i maksymalna długość** pokazują, czy fragmenty mieszczą się w limitach modelu osadzającego (np. `text-embedding-3-small` obsługuje do 8191 tokenów).
- Gdyby któryś fragment był za długi, należałoby go dodatkowo podzielić. W naszym regulaminie najdłuższy fragment to pojedynczy, rozbudowany ustęp — mieści się bezpiecznie w limitach.


In [ ]:
import json

sciezka_wynikowa = 'fragmenty_regulaminu.json'

with open(sciezka_wynikowa, 'w', encoding='utf-8') as f:
    json.dump(fragmenty, f, ensure_ascii=False, indent=2)

print(f'Zapisano {len(fragmenty)} fragmentów do pliku: {sciezka_wynikowa}')
print('Struktura pojedynczego fragmentu:')
print(json.dumps(fragmenty[0], ensure_ascii=False, indent=2))


### Podsumowanie etapu

W tym notebooku:

1. **Wydobyliśmy i oczyściliśmy** tekst regulaminu z pliku PDF (`regulamin_oczyszczony.txt`), naprawiając m.in. przełamane wiersze i uszkodzone tabele.
2. **Podzieliliśmy tekst na fragmenty semantyczne** (1 ustęp = 1 fragment), przypisując metadane `paragraf`, `ustęp` i `zrodło` (np. `§ 30 ust. 6`).
3. **Policzyliśmy tokeny** każdego fragmentu, potwierdzając, że mieszczą się w limitach modelu.
4. **Zapisaliśmy wyniki** do `fragmenty_regulaminu.json`.

Wykryliśmy i obsłużyliśmy **anomalię w § 30** (numeracja ustępów zaczyna się od 6) oraz **paragrafy bez numerowanych ustępów** (np. § 1), oznaczone jako ustęp 1.

Tak przygotowane fragmenty są gotowe do kolejnego etapu: osadzania (embedding) i budowy indeksu wektorowego.
